In [1]:
import sys; sys.path.append('../..')
import random
import pyzx as zx
from pyzx.drawing import draw

In [2]:
import logging

logger = logging.getLogger()
logger.setLevel(logging.INFO)

In [3]:
from pyzx.graph.base import BaseGraph
from pyzx.simplify import to_graph_like


def generate_graph(num_qubits: int, depth: int) -> "BaseGraph":
    random.seed(1341)
    g = zx.generate.cliffordT(qubits=num_qubits, depth=depth)
    to_graph_like(g)

    return g

### Heuristic simplification
- When simplifying ZX-diagrams with T-spiders, simplification routines like full_reduce lead to a very high two-qubit gate count.
- When using heuristic-based approaches we can circumvent the problem to some extent leading to better overall circuit cost after optimization

In [4]:
random.seed(1343)
g = zx.generate.cliffordT(qubits=5, depth=50, p_t=0.3, p_cnot=0.5)
c = zx.Circuit.from_graph(g)
c = zx.optimize.basic_optimization(c.split_phase_gates()).split_phase_gates()
print(c.stats())

Circuit  on 5 qubits with 50 gates.
        9 is the T-count
        41 Cliffords among which
        25 2-qubit gates (25 CNOT, 0 other) and
        4 Hadamard gates.


In [5]:
g = c.to_graph()
g_tele = zx.simplify.teleport_reduce(g)
g_tele.track_phases = False
print(zx.Circuit.from_graph(g).split_phase_gates().stats())

Circuit  on 5 qubits with 50 gates.
        9 is the T-count
        41 Cliffords among which
        25 2-qubit gates (25 CNOT, 0 other) and
        4 Hadamard gates.


In [6]:
g_full = g_tele.copy()
zx.simplify.full_reduce(g_full)
print(zx.extract_circuit(g_full.copy()).stats())

Circuit  on 5 qubits with 62 gates.
        9 is the T-count
        53 Cliffords among which
        24 2-qubit gates (2 CNOT, 22 other) and
        24 Hadamard gates.


The `greedy_simp` function is used to simplify a ZX-diagram by applying a greedy simplification algorithm to its neighbors. This function is part of the `pyzx.simplify` module in the `pyzx` library.

### Syntax
```python
zx.simplify.greedy_simp(
    graph: BaseGraph,
    max_vertex_index: Any | None = None,
    threshold: int = 1,
    lookahead: int = 0,
    use_yz_phase_gadgets: bool = False,
    use_xz_phase_gadgets: bool = False,
    flow_function: FilterFlowFunc = FilterFlowFunc.NONE,
) -> None
```

### Parameters
- `graph`: The ZX-diagram to be simplified.
- `max_vertex_index`: The maximum index allowed for matches. Algorithm will stop if no matches are found with index less than this value.
- `threshold`: The threshold for matches to be considered for simplification. If the heuristic value of a match is higher than or equal to the threshold, the match will be applied. The default value is 1.
- `lookahead`: The number of future matches to consider in the desicion of which match to apply. The default value is 0.
- `use_yz_phase_gadgets`: A boolean flag indicating whether to use YZ-phase gadgets during simplification. If set to `True`, `flow_function` is set to `FilterFlowFunc.G_FLOW_PRESERVING_GADGETS`.
- `use_xz_phase_gadgets`: A boolean flag indicating whether to use XZ-phase gadgets during simplification. If set to `True`, `flow_function` is set to `FilterFlowFunc.G_FLOW_PRESERVING_GADGETS`.
- `flow_function`: The flow function to use during simplification. The default value is `FilterFlowFunc.NONE`.

In [7]:
g_greedy = g_tele.copy()
zx.simplify.greedy_simp(g_greedy, lookahead=1, threshold=1, use_xz_phase_gadgets=True, use_yz_phase_gadgets=True)

print(f"graph is equal to input: {zx.compare_tensors(g_greedy, g_tele)}")

print(zx.extract_circuit(g_greedy.copy()).stats())

/home/wanja/Dokumente/Muniqc/pyzx-heuristics/demos/heuristic_demos/../../pyzx/heuristics/simplification.py:929: UserWarning: Phase gadgets require a flow function. Using G_FLOW_PRESERVING_GADGET function.
  warnings.warn(f"Phase gadgets require a flow function. Using {FilterFlowFunc(self.flow_function).name} function.")
  0%|          | 0/10 [00:00<?, ?it/s]

100%|██████████| 10/10 [00:00<00:00, 387.32it/s]
INFO:root:Applied match #1: (76, 77), (5, None, None)
INFO:root:Applied match #2: (39, 50), (6, None, None)
100%|██████████| 9/9 [00:00<00:00, 271.54it/s]
INFO:root:Applied match #3: (28, 33), (3, None, None)
INFO:root:Applied match #4: (48, 51), (2, None, None)
100%|██████████| 8/8 [00:00<00:00, 154.83it/s]
INFO:root:Applied match #5: (29, 35), (2, None, None)
INFO:root:Applied match #6: (72, 79), (1, None, None)
100%|██████████| 8/8 [00:00<00:00, 162.26it/s]
INFO:root:Applied match #7: (69,), (1.0, [55, 78], None)
INFO:root:Applied match #8: (55, 56), (2, None, None)
100%|██████████| 7/7 [00:00<00:00, 196.39it/s]
INFO:root:Applied match #9: (19, 20), (1, None, None)
INFO:root:Applied match #10: (34,), (1.0, [45, 30], None)
100%|██████████| 7/7 [00:00<00:00, 296.62it/s]
INFO:root:Applied match #11: (12,), (1.0, [10, 13], None)
INFO:root:Applied match #12: (13, 23), (2, None, None)
100%|██████████| 6/6 [00:00<00:00, 118.91it/s]
INFO:root

graph is equal to input: True
Circuit  on 5 qubits with 60 gates.
        9 is the T-count
        51 Cliffords among which
        21 2-qubit gates (2 CNOT, 19 other) and
        25 Hadamard gates.


In [8]:
draw(g_greedy)

The `greedy_simp_neighbors` function uses matches employing neighbor unfusion. This requires the FlowFunction to be set to `FilterFlowFunc.G_FLOW_PRESERVING` if gadgets are used it is set to `FilterFlowFunc.G_FLOW_PRESERVING_GADGETS`.

In [9]:
g_nu = g_tele.copy()
zx.simplify.greedy_simp_neighbors(g_nu, lookahead=1, threshold=1, use_xz_phase_gadgets=True, use_yz_phase_gadgets=True)

print(f"graph is equal to input: {zx.compare_tensors(g_nu, g_tele)}")

print(zx.extract_circuit(g_nu.copy()).stats())

/home/wanja/Dokumente/Muniqc/pyzx-heuristics/demos/heuristic_demos/../../pyzx/heuristics/simplification.py:921: UserWarning: Neighbor unfusion with phase gadgets requires a flow function. Using G_FLOW_PRESERVING_GADGET function.
  warnings.warn(f"Neighbor unfusion with phase gadgets requires a flow function. Using {FilterFlowFunc(self.flow_function).name} function.")
  0%|          | 0/16 [00:00<?, ?it/s]

100%|██████████| 16/16 [00:00<00:00, 127.12it/s]
INFO:root:Applied match #1: (76, 77), (5, None, None)
INFO:root:Applied match #2: (39, 50), (6, None, None)
100%|██████████| 14/14 [00:00<00:00, 141.54it/s]
INFO:root:Applied match #3: (28, 33), (3, None, None)
INFO:root:Applied match #4: (48, 51), (2, None, None)
100%|██████████| 14/14 [00:00<00:00, 346.15it/s]
INFO:root:Applied match #5: (29, 35), (2, None, None)
INFO:root:Applied match #6: (72, 79), (1, None, None)
100%|██████████| 14/14 [00:00<00:00, 219.49it/s]
INFO:root:Applied match #7: (69,), (1.0, [55, 78], None)
INFO:root:Applied match #8: (55, 56), (2, None, None)
100%|██████████| 13/13 [00:00<00:00, 336.34it/s]
INFO:root:Applied match #9: (12,), (1.0, [10, 13], None)
INFO:root:Applied match #10: (34,), (1.0, [45, 30], None)
100%|██████████| 13/13 [00:00<00:00, 484.23it/s]
INFO:root:Applied match #11: (19, 20), (1, None, None)
INFO:root:Applied match #12: (13, 23), (2, None, None)
100%|██████████| 13/13 [00:00<00:00, 410.90it/

graph is equal to input: True
Circuit  on 5 qubits with 54 gates.
        9 is the T-count
        45 Cliffords among which
        16 2-qubit gates (0 CNOT, 16 other) and
        24 Hadamard gates.


In [10]:
draw(g_nu)